# Portfolio Risk and Performance Measurement

This notebook measures portfolio performance, downside risk, and Value at Risk model quality for a selected equity universe.

The workflow is intentionally simple and reproducible:

1. Download adjusted close prices.
2. Convert prices to daily returns.
3. Estimate in-sample asset risk and performance metrics.
4. Build three portfolio strategies.
5. Compare realized out-of-sample performance.
6. Backtest 1% historical VaR.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yfinance as yf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.metrics import historical_var_backtest, performance_summary, summarize_assets
from src.portfolio import (
    build_strategy_returns,
    equal_weights,
    inverse_expected_shortfall_weights,
    top_assets_by_return,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:.4f}".format

## 2. Parameters

The dates below use 2022 as the in-sample period and 2023-2024 as the out-of-sample evaluation window.

In [ ]:
TICKERS = ["META", "AAPL", "TGT", "EXPD", "UNH", "SBUX", "ZBRA", "MRNA", "ACN", "TSN"]
START_DATE = "2022-01-01"
END_DATE = "2025-01-01"
IN_SAMPLE_END = "2022-12-31"
OUT_SAMPLE_START = "2023-01-01"
RISK_FREE_RATE = 0.03

## 3. Download Prices

In [ ]:
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)

if raw.empty:
    raise RuntimeError("No market data was downloaded. Check the internet connection or ticker list.")

prices = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw
prices = prices.dropna(how="all").ffill().dropna(axis=1)
returns = prices.pct_change().dropna()

print(f"Downloaded {prices.shape[0]} price rows for {prices.shape[1]} assets.")
prices.tail()

## 4. Price and Return Overview

In [ ]:
normalized_prices = prices / prices.iloc[0]
ax = normalized_prices.plot(figsize=(12, 6), linewidth=1.5)
ax.set_title("Normalized Price Performance")
ax.set_ylabel("Growth of $1")
ax.legend(loc="upper left", ncol=2)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(returns.corr(), cmap="vlag", center=0, annot=True, fmt=".2f", square=True)
plt.title("Daily Return Correlation")
plt.show()

## 5. In-Sample Asset Metrics

The in-sample period is used for ranking assets and constructing portfolio weights.

In [ ]:
in_sample = returns.loc[:IN_SAMPLE_END]
out_sample = returns.loc[OUT_SAMPLE_START:]

asset_summary = summarize_assets(in_sample, risk_free_rate=RISK_FREE_RATE)
asset_summary

## 6. Portfolio Construction

In [ ]:
top3 = top_assets_by_return(in_sample, n=3)
exclude_top3 = [ticker for ticker in in_sample.columns if ticker not in top3]
optimized_assets = asset_summary.head(5).index.tolist()

strategy_weights = {
    "Top 3 Equal Weight": equal_weights(top3),
    "Exclude Top 3 Equal Weight": equal_weights(exclude_top3),
    "Optimized 5 Inverse ES": inverse_expected_shortfall_weights(in_sample, optimized_assets),
}

weights_table = pd.DataFrame(strategy_weights).fillna(0)
weights_table

## 7. Out-of-Sample Portfolio Performance

In [ ]:
portfolio_returns = build_strategy_returns(out_sample, strategy_weights)
portfolio_summary = pd.DataFrame(
    {
        name: performance_summary(series, risk_free_rate=RISK_FREE_RATE)
        for name, series in portfolio_returns.items()
    }
).T
portfolio_summary

In [ ]:
wealth = (1 + portfolio_returns).cumprod()
ax = wealth.plot(figsize=(12, 6), linewidth=2)
ax.set_title("Out-of-Sample Portfolio Wealth")
ax.set_ylabel("Growth of $1")
plt.show()

## 8. Historical VaR Backtest

A 1% VaR model should produce exceptions on roughly 1% of days. Because the sample is finite and market regimes change, the realized exception rate will not match exactly.

In [ ]:
backtest_rows = []
backtests = {}

for name, series in portfolio_returns.items():
    bt = historical_var_backtest(series, window=250, level=0.01)
    backtests[name] = bt
    backtest_rows.append(
        {
            "Strategy": name,
            "Observations": len(bt),
            "Exceptions": int(bt["Exception"].sum()),
            "Expected Exceptions": len(bt) * 0.01,
            "Actual Exception Rate": bt.attrs["actual_exception_rate"],
            "Expected Exception Rate": bt.attrs["expected_exception_rate"],
        }
    )

backtest_summary = pd.DataFrame(backtest_rows).set_index("Strategy")
backtest_summary

In [ ]:
fig, axes = plt.subplots(len(backtests), 1, figsize=(12, 4 * len(backtests)), sharex=True)
if len(backtests) == 1:
    axes = [axes]

for ax, (name, bt) in zip(axes, backtests.items()):
    ax.plot(bt.index, bt["Realized Return"], label="Realized Return", linewidth=1)
    ax.plot(bt.index, bt["VaR"], label="1% Historical VaR", linewidth=1.5)
    exceptions = bt[bt["Exception"]]
    ax.scatter(exceptions.index, exceptions["Realized Return"], color="red", s=25, label="Exception")
    ax.set_title(name)
    ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

## 9. Conclusion Template

Use the tables above to discuss:

- which portfolio had the strongest risk-adjusted performance,
- whether higher returns came with materially higher drawdowns,
- whether the VaR model produced an exception rate close to the expected 1%,
- why historical VaR may fail during periods that differ from the estimation window.